# Yankari Game Reserve — Data Exploration
Week 1 deliverable: verify joined raw data quality before labeling and modelling.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data_pipeline.config import DATA_RAW

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
PLOTS = '../results/plots'
import os; os.makedirs(PLOTS, exist_ok=True)

## Load joined Yankari raw data

In [ ]:
df = pd.read_csv(DATA_RAW / 'yankari_raw_2020_2025.csv', index_col='date', parse_dates=True)
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
df.head()

## Missing data summary

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print('Missing fraction per column:')
print(missing[missing > 0].apply(lambda x: f'{x:.1%}').to_string())

## Rainfall over time (Open-Meteo)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
rain_col = 'om_precipitation_sum' if 'om_precipitation_sum' in df.columns else [c for c in df.columns if 'precip' in c.lower()][0]
ax.bar(df.index, df[rain_col], width=1, color='steelblue', alpha=0.7)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.set_ylabel('Precipitation (mm/day)')
ax.set_title('Daily Rainfall — Yankari Game Reserve 2020–2025')
plt.tight_layout()
plt.savefig(f'{PLOTS}/yankari_rainfall.png')
plt.show()

## NDVI over time (Sentinel-2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ndvi = df['ndvi_s2'].dropna()
ax.scatter(ndvi.index, ndvi.values, s=6, color='forestgreen', alpha=0.6, label='NDVI observation')
if len(ndvi) >= 10:
    rolling = ndvi.rolling(10, center=True).mean()
    ax.plot(rolling.index, rolling.values, color='darkgreen', lw=1.5, label='10-obs rolling mean')
ax.axhline(0.3, color='orange', lw=1, ls='--', label='Degradation threshold (0.3)')
ax.set_ylim(-0.1, 1.0)
ax.set_ylabel('NDVI')
ax.set_title('Sentinel-2 NDVI — Yankari Game Reserve 2020–2025')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{PLOTS}/yankari_ndvi.png')
plt.show()

## Fire detections over time (FIRMS VIIRS)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
if 'firms_count' in df.columns:
    fire = df['firms_count'].fillna(0)
    ax.bar(df.index, fire, width=1, color='firebrick', alpha=0.8)
    ax.set_ylabel('FIRMS detections / day')
else:
    ax.text(0.5, 0.5, 'firms_count column not found', ha='center', transform=ax.transAxes)
ax.set_title('Daily Fire Detections — Yankari Game Reserve 2020–2025')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.savefig(f'{PLOTS}/yankari_fire_counts.png')
plt.show()

## Temperature — max daily (drought indicator)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
tmax_col = next((c for c in df.columns if 'temperature_2m_max' in c or 't2m_max' in c.lower()), None)
if tmax_col:
    ax.plot(df.index, df[tmax_col], lw=0.6, color='darkorange', alpha=0.8)
    ax.axhline(35, color='red', lw=1, ls='--', label='35 °C stress threshold')
    ax.set_ylabel('T_max (°C)')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'T_max column not found', ha='center', transform=ax.transAxes)
ax.set_title('Daily Maximum Temperature — Yankari Game Reserve 2020–2025')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.savefig(f'{PLOTS}/yankari_tmax.png')
plt.show()

## Summary statistics

In [ ]:
climate_cols = [c for c in df.columns if c.startswith('om_') or c.startswith('power_')]
summary = df[climate_cols].describe().T[['count', 'mean', 'std', 'min', 'max']]
summary['missing%'] = (1 - df[climate_cols].notna().mean()) * 100
summary.round(2)